# eval_base: evaluate the original Qwen2.5-3B-Instruct on Colab (first 50 MedQA questions, for debugging)

Steps: check GPU -> set path variables -> clone the repo -> install dependencies -> mount Google Drive -> run unit tests -> look at data samples -> evaluate -> compute metrics -> save to Drive.

**Before running**: pick a GPU under Runtime -> Change runtime type (a T4 is fine).

In [ ]:
!nvidia-smi

## 1. Variables to edit

In [ ]:
# ======== edit as needed ========
REPO_URL = "https://github.com/skjdsk/medqa-calibration.git"   # the project repo
SUBDIR = "."   # folder in the repo where the eval code lives (change if the folder gets renamed)
DRIVE_OUT_DIR = "/content/drive/MyDrive/medqa-calibration-results"          # shared Drive folder (for a shared folder, first "Add shortcut to My Drive" in Drive)
DATASET = "medqa"        # medqa / medmcqa / pubmedqa (several OK, separated by spaces)
NUM_OPTIONS = 4          # number of MedQA options: 4 or 5
N_DEBUG = 50             # only run the first N questions; set to None for the full test set
ADAPTER = None           # LoRA adapter path when evaluating a trained model, e.g. "/content/drive/MyDrive/adapters/sft_seed1"
MODEL_NAME = "base"      # model name written to the result files, e.g. "sft_seed1"
BATCH_SIZE = 8           # 8 is usually fine on a T4; PubMedQA abstracts are long, so drop to 4 if you run out of memory
# ==========================
REPO_DIR = "/content/project"            # where the repo gets cloned on Colab
CODE_DIR = f"{REPO_DIR}/{SUBDIR}"        # every command runs in this folder
LOCAL_OUT = "/content/results"

## 2. Get the code, install dependencies

In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
# go into the eval code folder: python -m eval.xxx has to run from here to find the eval package
%cd {CODE_DIR}
!pip install -q -r requirements.txt


## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
print("results will be saved to:", DRIVE_OUT_DIR)

## 4. (Optional) log in to Hugging Face
Anonymous downloads are rate-limited; if downloads are slow or you get a 429 error, add a secret called `HF_TOKEN` under the key icon on the left in Colab and run this cell.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("HF_TOKEN not set (fine to ignore):", type(e).__name__)

## 5. Unit tests: make sure the metric formulas are right (a few seconds, no GPU)

In [ ]:
!python -m pytest eval/test_metrics.py -q

## 6. Look at data samples first (no model loaded)
**Check carefully**: field names, that all options are there, that the gold letter matches the answer text, and that the answer distribution looks reasonable.

In [ ]:
!python -m eval.data_loaders --dataset medqa --num_options {NUM_OPTIONS} --n 3

## 7. Evaluate
The output prints the token ids for each letter, the full prompt for the first question, accuracy, and mean letter_mass (should be close to 1).

In [ ]:
n_arg = f"--n {N_DEBUG}" if N_DEBUG else ""
adapter_arg = f'--adapter "{ADAPTER}"' if ADAPTER else ""
!python -m eval.evaluate --dataset {DATASET} --num_options {NUM_OPTIONS} {n_arg} {adapter_arg} \
    --model_name {MODEL_NAME} --batch_size {BATCH_SIZE} --out_dir {LOCAL_OUT}/preds --check_batch_consistency

## 8. Compute summary metrics and draw reliability diagrams (only reads the CSVs, no model needed)

In [ ]:
!python -m eval.metrics --pred_csv "{LOCAL_OUT}/preds/preds_*.csv" --out_dir {LOCAL_OUT}/metrics

## 9. Copy to Drive and show the results
Each model's results go in `DRIVE_OUT_DIR/<model name>/` so they don't overwrite each other, with two subfolders: `preds/` (per-question CSVs and meta) and `metrics/` (summary tables and reliability diagrams).

In [ ]:
import shutil, glob
import pandas as pd
from IPython.display import Image, display

dest = os.path.join(DRIVE_OUT_DIR, MODEL_NAME)
shutil.copytree(LOCAL_OUT, dest, dirs_exist_ok=True)
print("copied to:", dest)
for f in sorted(glob.glob(os.path.join(dest, "**", "*"), recursive=True)):
    print("  ", f)

display(pd.read_csv(os.path.join(LOCAL_OUT, "metrics", "summary.csv")).round(4))
for png in sorted(glob.glob(os.path.join(LOCAL_OUT, "metrics", "reliability_*.png"))):
    display(Image(png, width=420))

## Later: summarize all 7 models
Once every model is done, you can compute one combined table from all the CSVs on Drive:
```
%cd {CODE_DIR}
!python -m eval.metrics --pred_csv "/content/drive/MyDrive/medqa-calibration-results/*/preds/preds_*.csv" --out_dir /content/drive/MyDrive/medqa-calibration-results/all_models
```